In [9]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# 设置路径
input_path = r"F:\TUB\greenlandProject\ERA5LAND\2020snowcsde.nc"
output_folder = r"F:\TUB\greenlandProject\ERA5LAND\2020"
os.makedirs(output_folder, exist_ok=True)

# 打开数据集
ds = xr.open_dataset(input_path)

# 转换经度到 [-180, 180]，并排序
ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
ds = ds.sortby('longitude')  # 确保经度升序排列

# 裁剪格陵兰地区（注意纬度从高到低）
ds_crop = ds.sel(
    longitude=slice(-75, -10),
    latitude=slice(85, 55)
)

# 提取变量
snowc = ds_crop['snowc']
sde = ds_crop['sde']
valid_time = ds_crop['valid_time']


# 比较 1 月和 8 月的 sde 是否一致
sde_jan = sde.sel(valid_time=valid_time[0])
sde_aug = sde.sel(valid_time=valid_time[3])
is_equal = np.array_equal(sde.values, sde_aug.values)
print(f"1月和8月的 sde 是否完全一致: {is_equal}")





# 比较 1 月和 8 月的 snowc
snowc_jan = snowc.sel(valid_time=valid_time[0])
snowc_aug = snowc.sel(valid_time=valid_time[7])

# 比较差异
diff_mask = (snowc_jan != snowc_aug)
num_total = diff_mask.size
num_diff = diff_mask.sum().item()
diff_ratio = num_diff / num_total

print(f"1月和8月的 snowc 是否完全一致: {num_diff == 0}")
print(f"不一致的格网数: {num_diff} / {num_total} = {diff_ratio:.2%}")

# 打印几个不一致的格网点（带经纬度和值）
diff_indices = np.where(diff_mask)
lat_vals = ds_crop.latitude.values
lon_vals = ds_crop.longitude.values

print("\n前几个不一致的格网点（纬度, 经度, 1月值, 8月值）:")
for i in range(min(10, len(diff_indices[0]))):
    y, x = diff_indices[0][i], diff_indices[1][i]
    lat = lat_vals[y]
    lon = lon_vals[x]
    val_jan = snowc_jan.values[y, x]
    val_aug = snowc_aug.values[y, x]
    print(f"({lat:.2f}, {lon:.2f}) -> Jan: {val_jan}, Aug: {val_aug}")


# 经纬度网格（用于绘图）
lon2d, lat2d = np.meshgrid(ds_crop.longitude, ds_crop.latitude)

# 逐月绘图
for i, time_val in enumerate(valid_time.values):
    date_str = np.datetime_as_string(time_val, unit='M')  # 如 "2020-01"
    snowc_month = snowc.sel(valid_time=time_val)
    sde_month = sde.sel(valid_time=time_val)

    for var, data in zip(['snowc', 'sde'], [snowc_month, sde_month]):
        plt.figure(figsize=(10, 6))
        mesh = plt.pcolormesh(lon2d, lat2d, data, cmap='viridis', shading='auto')
        plt.colorbar(mesh, label=var)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title(f"{date_str} {var}")
        output_file = os.path.join(output_folder, f"{date_str.replace('-', '')}{var}.png")
        plt.savefig(output_file, dpi=300, bbox_inches='tight')
        plt.close()





1月和8月的 sde 是否完全一致: False
1月和8月的 snowc 是否完全一致: False
不一致的格网数: 133602 / 195951 = 68.18%

前几个不一致的格网点（纬度, 经度, 1月值, 8月值）:
(85.00, -75.00) -> Jan: nan, Aug: nan
(85.00, -74.90) -> Jan: nan, Aug: nan
(85.00, -74.80) -> Jan: nan, Aug: nan
(85.00, -74.70) -> Jan: nan, Aug: nan
(85.00, -74.60) -> Jan: nan, Aug: nan
(85.00, -74.50) -> Jan: nan, Aug: nan
(85.00, -74.40) -> Jan: nan, Aug: nan
(85.00, -74.30) -> Jan: nan, Aug: nan
(85.00, -74.20) -> Jan: nan, Aug: nan
(85.00, -74.10) -> Jan: nan, Aug: nan
